# ДЗ 16. Pandas: первая таблица, столбцы и фильтры
**Киберформула · Python и анализ данных**

Файлы: `orders.csv` (чистая выгрузка заказов) и `orders_raw.csv` (та же выгрузка до чистки).

| Уровень | Задачи |
|---|---|
| **База** | 1–3 |
| **Челлендж** | 4–7 |
| **Про** | 8–9 |

**Правило:** после каждого ответа напишите одну фразу-вывод комментарием `# Вывод: ...`. Число без вывода — половина ответа.
Если застряли, откройте справочник урока 16.

In [2]:
# Загрузка файлов: в Colab нажмите «Выбрать файлы» и укажите orders.csv и orders_raw.csv
import os
need = [f for f in ["orders.csv", "orders_raw.csv"] if not os.path.exists(f)]
if need:
    try:
        from google.colab import files
        files.upload()
    except ImportError:
        print("Положите orders.csv и orders_raw.csv в одну папку с ноутбуком")
for f in ["orders.csv", "orders_raw.csv"]:
    print(f, "на месте:", os.path.exists(f))

import pandas as pd

orders.csv на месте: True
orders_raw.csv на месте: True


## База

### Задача 1. Первый взгляд
Загрузите `orders.csv` в переменную `df`.
1. Сколько в таблице строк и столбцов?
2. Сколько заказов из Караганды?
3. Какие товары продаются в категории «Аксессуары» и сколько их разных?

In [3]:
# Задача 1
import pandas as pd

df = pd.read_csv('orders.csv')

print("Строк и столбцов:", df.shape)

print("Заказов из Караганды:",
      df[df['city'] == 'Караганда'].shape[0])

print("Товары категории Аксессуары:",
      df.loc[df['category'] == 'Аксессуары', 'product'].unique())

print("Количество разных товаров:",
      df.loc[df['category'] == 'Аксессуары', 'product'].nunique())

Строк и столбцов: (500, 9)
Заказов из Караганды: 72
Товары категории Аксессуары: <StringArray>
['Клавиатура Logitech K120',    'Наушники JBL Tune 520',
  'Power bank Xiaomi 20000',       'Мышь Logitech M185']
Length: 4, dtype: str
Количество разных товаров: 4


### Задача 2. Дорогие заказы Шымкента
Добавьте столбец `revenue` = `price` × `qty`. Выведите 5 самых дорогих **оплаченных** заказов из Шымкента: только столбцы `date`, `product`, `qty`, `revenue`.

In [4]:
# Задача 2
df['revenue'] = df['price'] * df['qty']

res_pd = (df.loc[
    (df['city'] == 'Шымкент') &
    (df['status'] == 'оплачен'),
    ['date', 'product', 'qty', 'revenue']
]
.sort_values('revenue', ascending=False)
.head(5))

res_pd

,date,product,qty,revenue
202,2026-06-30,ASUS VivoBook 15,2,638000
162,2026-06-05,Lenovo IdeaPad 5,2,578000
219,2026-07-07,MacBook Air M3,1,549000
172,2026-06-13,MacBook Air M3,1,549000
336,2026-08-24,MacBook Air M3,1,549000


### Задача 3. Покупают помногу?
1. Сколько заказов, в которых купили **2 штуки или больше**? Какая это доля всех заказов в %?
2. Какой товар чаще всего покупают сразу **по 3 штуки**?

In [6]:
# Задача 3
count = df[df['qty'] >= 2].shape[0]

percent = count / len(df) * 100

print("Количество заказов:", count)
print("Доля заказов:", round(percent, 2), "%")

res = (df[df['qty'] == 3]['product']
       .value_counts()
       .head(1))

print(res)

Количество заказов: 116
Доля заказов: 23.2 %
product
Наушники JBL Tune 520    13
Name: count, dtype: int64


## Челлендж

### Задача 4. Алматы против Астаны
Для **оплаченных** заказов посчитайте средний чек (`revenue`) и медиану отдельно для Алматы и для Астаны.
Средние получатся почти одинаковые, а медианы — нет. Объясните, что это говорит о покупках в двух городах.

In [7]:
# Задача 4
df['revenue'] = df['price'] * df['qty']

res = df[
    (df['status'] == 'оплачен') &
    (df['city'].isin(['Алматы', 'Астана']))
]

result = res.groupby('city')['revenue'].agg(['mean', 'median'])

print(result)

                 mean    median
city                           
Алматы  187184.426230  129000.0
Астана  188079.545455  159000.0


### Задача 5. Весна против лета
Сравните **весну** (март–май) и **лето** (июнь–август) по оплаченным заказам:
выручка, изменение в %, число заказов.

Подсказка: `month = df["date"].str[:7]` и `month.isin(["2026-03", "2026-04", "2026-05"])`.

In [ ]:
# Задача 5
df['revenue'] = df['price'] * df['qty']

paid = df[df['status'] == 'оплачен'].copy()

month = paid['date'].str[:7]

spring = paid[month.isin(['2026-03', '2026-04', '2026-05'])]
summer = paid[month.isin(['2026-06', '2026-07', '2026-08'])]

spring_revenue = spring['revenue'].sum()
summer_revenue = summer['revenue'].sum()

change = (summer_revenue - spring_revenue) / spring_revenue * 100 if spring_revenue != 0 else float('nan')

print("Весна:")
print("Выручка:", spring_revenue)
print("Количество заказов:", len(spring))

print("Лето:")
print("Выручка:", summer_revenue)
print("Количество заказов:", len(summer))

print("Изменение выручки:", round(change, 2), "%")

Весна:
Выручка: 28450500
Количество заказов: 142
Лето:
Выручка: 35892500
Количество заказов: 188
Изменение выручки: 26.16 %


### Задача 6. Где чаще отменяют?
Посчитайте долю отменённых заказов (в %) в **каждой** категории.
В какой категории отменяют чаще всего, в какой реже всего? Во сколько раз отличается?

Подсказка: цикл `for cat in df["category"].unique():`.

In [10]:
# Задача 6
results = []


for cat in df['category'].unique():
    data = df[df['category'] == cat]
    total = len(data)
    cancelled = len(data[data['status'] == 'отменён'])
    percent = cancelled / total * 100
    results.append([cat, percent])

result = pd.DataFrame(results, columns=['category', 'cancel_percent'])
result = result.sort_values('cancel_percent', ascending=False)

print(result)
print("Чаще всего отменяют:", result.iloc[0]['category'])
print("Реже всего отменяют:", result.iloc[-1]['category'])

max_percent = result['cancel_percent'].max()
min_percent = result['cancel_percent'].min()

if min_percent > 0:
    print("Разница в разах:", round(max_percent / min_percent, 2))
else:
    print("Разницу в разах нельзя вычислить: минимальная доля равна 0%")

     category  cancel_percent
0   Смартфоны       12.213740
3    Мониторы       12.000000
1  Аксессуары        8.771930
4    Ноутбуки        6.185567
2    Планшеты        3.921569
Чаще всего отменяют: Смартфоны
Реже всего отменяют: Планшеты
Разница в разах: 3.11


### Задача 7. Разведка `orders_raw.csv`
Загрузите сырой файл в переменную `raw`. Ничего не исправляйте, только считайте.
1. Сколько в нём строк и сколько разных `order_id`? Сколько лишних строк?
2. Сколько разных написаний города? Выпишите все варианты Алматы.
3. Сколько заказов найдёт `raw[raw["city"] == "Алматы"]`? А сколько заказов из Алматы в чистой таблице?

In [ ]:
# Задача 7

## Про

### Задача 8. Какие бывают «цены»
В `raw["price"]` записаны 518 значений. Разложите их по видам и посчитайте, сколько в каждом:

| Вид | Пример | Как найти |
|---|---|---|
| только цифры | `99000` | `raw["price"].str.isdigit() == True` |
| со знаком ₸ | `99000 ₸` | `.str.contains("₸", na=False)` |
| с `.0` на конце | `99000.0` | `.str.endswith(".0", na=False)` |
| с пробелом между цифрами | `99 000` | придумайте сами |
| слова вместо цены | ? | найдите через `value_counts()` или `unique()` |
| пропуск | `NaN` | `.isna()` |

Проверьте, что сумма всех видов — ровно 518. Если нет, какое-то значение попало в два вида или не попало никуда.

In [ ]:
# Задача 8

### Задача 9. Популярный ≠ прибыльный?
1. Какой товар **заказывают чаще всего** (по числу заказов)?
2. Какой товар принёс **больше всего выручки** по оплаченным заказам? Найдите его циклом по `df["product"].unique()`: считайте выручку каждого товара и запоминайте лучший.
3. Во сколько раз выручка самого прибыльного товара больше выручки самого популярного?

In [ ]:
# Задача 9